# Underground analysis — upgrade cells

Companion to `Underground_analysis.ipynb`. It reads the `groundwater_cleaned.csv` that notebook exports (so run that one first) and adds:

1. **Setup + county-name join check**
2. **3D H3 hexagon maps** — record density and median well depth, with thin-sample hexes greyed out
3. **Wells hanging below real terrain** — depth drawn *downward* from ground level
4. **Monitoring-gap surface** — height = distance to the nearest inventory record (an honest alternative to interpolating depth)
5. **GPU time filter (lonboard)** — instant month slider, cumulative or single month
6. **Cross-section along a transect**
7. **Depth distributions** by source type and county

`well_depth` is still construction depth, not water level, and `last_update` is still a record-update date. Nothing here changes that.

> Written without access to your data. Cells were syntax-checked but not executed. If a layer renders oddly, the comments flag the likeliest knob.

## 1. Setup and county-name join check

In [ ]:
# Manual alternative: %pip install -q h3 lonboard pyarrow pydeck plotly geopandas scipy ipywidgets

from pathlib import Path
import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
import pydeck as pdk
# Install optional GPU/geospatial dependencies into the active notebook kernel.
# This runs only when a dependency is missing.
RUNTIME_PACKAGES = {
    "h3": "h3",
    "lonboard": "lonboard",
    "pyarrow": "pyarrow",
}
missing_packages = [
    package_name
    for import_name, package_name in RUNTIME_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    print("Installing missing notebook packages:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing_packages
    ])

# pyright: reportMissingImports=false
import h3
from scipy.spatial import cKDTree
from IPython.display import display, HTML

PROJECT_DIR = Path.cwd()
CLEAN_CSV = PROJECT_DIR / "groundwater_cleaned.csv"
BOUNDARY_PATH = PROJECT_DIR / "gadm36_KEN_1.shp"

H3_RES = 5            # 5 ~ 250 km2 cells (national view). Try 6 for finer, 4 for coarser.
MIN_DEPTH_N = 5       # hexes/counties with fewer depth measurements are greyed out
H_MAX = 70_000        # tallest hex in metres (visual units; legend explains the mapping)
MAP_CENTER = {"lat": 0.15, "lon": 37.85}
DARK_STYLE = "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json"

df = pd.read_csv(CLEAN_CSV, parse_dates=["last_update"], low_memory=False)
df["has_depth"] = df["has_depth"].astype(bool)
df["depth_review_flag"] = df["depth_review_flag"].astype(bool)
df["height_value"] = df["well_depth"].where(df["has_depth"])
counties = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")

# --- County names: normalise spelling so the data joins to GADM -------------
def _norm(s):
    return "".join(ch for ch in str(s).lower() if ch.isalpha())

MANUAL_FIXES = {}   # e.g. {"Nairobi City": "Nairobi"} after you see the unmatched list below
lookup = {_norm(n): n for n in counties["NAME_1"]}
def to_gadm(v):
    if pd.isna(v):
        return None
    v = MANUAL_FIXES.get(v, v)
    return lookup.get(_norm(v))

df["county"] = df["admin_1"].map(to_gadm)
unmatched = df.loc[df["county"].isna() & df["admin_1"].notna(), "admin_1"].value_counts()
print(f"Records with a GADM county: {df['county'].notna().mean():.1%}")
if len(unmatched):
    print("Unmatched admin_1 values (add to MANUAL_FIXES):")
    print(unmatched.head(20))
print("GADM counties with no records:", sorted(set(counties["NAME_1"]) - set(df["county"].dropna())))

# --- Shared colour helpers ----------------------------------------------------
PLASMA = [(13, 8, 135), (126, 3, 168), (204, 71, 120), (248, 149, 64), (240, 249, 33)]
VIRIDIS = [(68, 1, 84), (59, 82, 139), (33, 145, 140), (94, 201, 98), (253, 231, 37)]
MAGMA = [(0, 0, 4), (81, 18, 124), (183, 55, 121), (252, 137, 97), (252, 253, 191)]
GREY = [90, 98, 108, 120]

def ramp(t, stops):
    t = float(np.clip(0.0 if pd.isna(t) else t, 0, 1))
    pos = np.linspace(0, 1, len(stops))
    return [int(np.interp(t, pos, [s[i] for s in stops])) for i in range(3)]

TYPE_RGB = {
    "Protected spring": (31, 164, 214), "Unprotected spring": (79, 209, 197),
    "Borehole or tubewell": (150, 110, 230), "Protected dug well": (244, 166, 50),
    "Unprotected dug well": (225, 82, 82),
}
def type_color(t, alpha=230):
    return list(TYPE_RGB.get(t, (145, 150, 160))) + [alpha]

def colorbar_html(title, subtitle, lo_label, hi_label, stops, extra=""):
    grad = ", ".join(f"rgb({r},{g},{b})" for r, g, b in stops)
    return f'''<div style="position:absolute;z-index:10;top:16px;left:16px;max-width:300px;padding:12px 14px;background:rgba(16,22,30,.88);color:#e8edf2;border-radius:8px;font:12px/1.4 Arial;box-shadow:0 3px 16px rgba(0,0,0,.4)">
<div style="font-size:14px;font-weight:700">{title}</div>
<div style="color:#9aa7b4;margin:3px 0 9px">{subtitle}</div>
<div style="height:10px;border-radius:5px;background:linear-gradient(90deg,{grad})"></div>
<div style="display:flex;justify-content:space-between;margin-top:3px"><span>{lo_label}</span><span>{hi_label}</span></div>
<div style="margin-top:8px;color:#9aa7b4">{extra}</div></div>'''

def show_and_export(deck, legend, filename):
    display(HTML(legend))
    display(deck)
    html = deck.to_html(as_string=True, notebook_display=False)
    Path(filename).write_text(html.replace("</body>", legend + "</body>"), encoding="utf-8")
    print("Saved", filename)

TOOLTIP_STYLE = {"backgroundColor": "rgba(18,27,36,.94)", "color": "white",
                 "borderRadius": "7px", "padding": "10px 12px"}

## 2. 3D H3 hexagon maps

Hexagons replace thousands of overlapping point columns, so no single record is over-emphasised and spatial coverage reads at a glance. Two views:

- **Density** — height and colour = number of records
- **Median well depth** — height and colour = median depth; hexes with fewer than `MIN_DEPTH_N` depth measurements are flat grey, so thin samples don't masquerade as findings

In [ ]:
df["hex"] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(df["latitude"], df["longitude"])]

hx = (df.groupby("hex")
        .agg(records=("original_id", "size"),
             n_depth=("has_depth", "sum"),
             median_depth=("height_value", "median"),
             median_elev=("elevation", "median"))
        .reset_index())
hx["depth_ok"] = hx["n_depth"] >= MIN_DEPTH_N

# Density encoding (sqrt so a few dense hexes don't flatten everything else)
t_c = np.sqrt(hx["records"] / hx["records"].max())
hx["h_count"] = 3000 + (H_MAX - 3000) * t_c
hx["c_count"] = [ramp(t, VIRIDIS) + [225] for t in t_c]

# Median depth encoding (robust 5-95% range)
d_lo, d_hi = hx.loc[hx["depth_ok"], "median_depth"].quantile([0.05, 0.95])
t_d = ((hx["median_depth"] - d_lo) / max(d_hi - d_lo, 1e-9)).clip(0, 1).fillna(0)
hx["h_depth"] = np.where(hx["depth_ok"], 4000 + (H_MAX - 4000) * t_d, 2000.0)
hx["c_depth"] = [ramp(t, PLASMA) + [225] if ok else GREY for t, ok in zip(t_d, hx["depth_ok"])]
hx["depth_label"] = [f"{v:,.1f} m" if ok else "too few measurements"
                     for v, ok in zip(hx["median_depth"], hx["depth_ok"])]
hx["elev_label"] = hx["median_elev"].map(lambda v: "n/a" if pd.isna(v) else f"{v:,.0f} m")
hx = hx.drop(columns=["median_depth", "median_elev"])   # keep the JSON NaN-free

def hex_deck(h_col, c_col, description):
    layers = [
        pdk.Layer("GeoJsonLayer", counties[["NAME_1", "geometry"]].__geo_interface__,
                  stroked=True, filled=False, get_line_color=[210, 220, 230, 90],
                  line_width_min_pixels=0.8),
        pdk.Layer("H3HexagonLayer", hx, get_hexagon="hex", get_fill_color=c_col,
                  get_elevation=h_col, elevation_scale=1, extruded=True, coverage=0.93,
                  pickable=True, auto_highlight=True,
                  material={"ambient": 0.45, "diffuse": 0.6, "shininess": 24}),
    ]
    tooltip = {"html": ("<b>{records}</b> records<br/>Depth measurements: {n_depth}<br/>"
                        "Median well depth: {depth_label}<br/>Median elevation: {elev_label}"),
               "style": TOOLTIP_STYLE}
    return pdk.Deck(layers=layers, map_style=DARK_STYLE, tooltip=tooltip, description=description,
                    initial_view_state=pdk.ViewState(latitude=MAP_CENTER["lat"], longitude=MAP_CENTER["lon"],
                                                     zoom=5.3, pitch=55, bearing=-14))

# Density
deck_count = hex_deck("h_count", "c_count", "Kenya groundwater inventory — record density")
legend = colorbar_html("Record density", f"H3 resolution {H3_RES}; height and colour = records",
                       "1", f"{int(hx['records'].max()):,} records", VIRIDIS)
show_and_export(deck_count, legend, PROJECT_DIR / "gw_hex_density_3d.html")

# Median well depth
deck_depth = hex_deck("h_depth", "c_depth", "Kenya groundwater inventory — median well depth")
legend = colorbar_html("Median well depth (not water level)",
                       f"Grey = fewer than {MIN_DEPTH_N} depth measurements",
                       f"{d_lo:,.0f} m", f"{d_hi:,.0f} m", PLASMA,
                       "Height and colour both scale with median constructed depth")
show_and_export(deck_depth, legend, PROJECT_DIR / "gw_hex_depth_3d.html")

## 3. Wells hanging below real terrain

Each well is a vertical line from ground level **down** to ground level minus `well_depth`. The terrain is a translucent 3D surface from open elevation tiles, so you can see how depths relate to relief.

- `VE` is vertical exaggeration, applied identically to terrain and well depth, so they stay in true proportion to each other. Raise it for flat areas.
- The map covers the entire Kenyan study area.
- Collar `elevation` is taken from your table (GPS-derived values can be noisy), with gaps filled by the relevant county median and then the national median. Check it before drawing conclusions.

In [ ]:
REGION = "Entire study area"
VE = 8                      # vertical exaggeration (terrain and depth together)

sub = df.copy()
sub["elev_m"] = sub["elevation"].where(sub["elevation"] > -50)
if sub["elev_m"].isna().all():
    raise ValueError("No usable elevation values in the study area.")
county_medians = sub.groupby("county")["elev_m"].transform("median")
sub["elev_m"] = sub["elev_m"].fillna(county_medians).fillna(sub["elev_m"].median())

# Wells flagged >1,000 m are excluded from the drawn lines (still counted in the table)
wells = sub[sub["has_depth"] & ~sub["depth_review_flag"]].copy()
wells["src"] = [[lo, la, e * VE] for lo, la, e in zip(wells["longitude"], wells["latitude"], wells["elev_m"])]
wells["tgt"] = [[lo, la, (e - d) * VE] for lo, la, e, d in zip(wells["longitude"], wells["latitude"],
                                                            wells["elev_m"], wells["well_depth"])]
wells["color"] = wells["Type"].map(lambda t: type_color(t, 235))
wells["depth_label"] = wells["well_depth"].map(lambda v: f"{v:,.1f} m")
wells["elev_label"] = wells["elev_m"].map(lambda v: f"{v:,.0f} m")
wells = wells[["name", "Type", "depth_label", "elev_label", "src", "tgt", "color"]]

collars = sub.copy()
collars["pos"] = [[lo, la, e * VE] for lo, la, e in zip(collars["longitude"], collars["latitude"], collars["elev_m"])]
collars["color"] = collars["Type"].map(lambda t: type_color(t, 255))
collars["depth_label"] = collars["height_value"].map(lambda v: "not recorded" if pd.isna(v) else f"{v:,.1f} m")
collars["elev_label"] = collars["elev_m"].map(lambda v: f"{v:,.0f} m")
collars = collars[["name", "Type", "depth_label", "elev_label", "pos", "color"]]

# Terrarium-encoded elevation tiles; the decoder is scaled by VE to exaggerate relief.
ELEV_DECODER = {"rScaler": 256 * VE, "gScaler": 1 * VE, "bScaler": VE / 256, "offset": -32768 * VE}

terrain = pdk.Layer(
    "TerrainLayer",
    elevation_decoder=ELEV_DECODER,
    texture="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    elevation_data="https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png",
    max_zoom=12, opacity=0.62,
    # depthMask False lets the below-ground lines show through the surface.
    # If the terrain looks speckled, delete this line and lower opacity instead.
    parameters={"depthMask": False},
)
well_lines = pdk.Layer("LineLayer", wells, get_source_position="src", get_target_position="tgt",
                       get_color="color", get_width=2, width_min_pixels=1.5, pickable=True)
collar_dots = pdk.Layer("ScatterplotLayer", collars, get_position="pos", get_fill_color="color",
                        get_radius=450, radius_min_pixels=2, radius_max_pixels=7, pickable=True)

lat_span = sub["latitude"].max() - sub["latitude"].min()
lon_span = sub["longitude"].max() - sub["longitude"].min()
zoom = float(np.clip(np.log2(360 / max(lat_span, lon_span, 0.05)) - 1.2, 4.5, 11))
view_lat = float((sub["latitude"].min() + sub["latitude"].max()) / 2)
view_lon = float((sub["longitude"].min() + sub["longitude"].max()) / 2)

deck_terrain = pdk.Deck(
    layers=[terrain, well_lines, collar_dots],
    initial_view_state=pdk.ViewState(latitude=view_lat, longitude=view_lon,
                                     zoom=zoom, pitch=68, bearing=20, max_pitch=85),
    map_style=None,
    tooltip={"html": "<b>{name}</b><br/>{Type}<br/>Well depth: {depth_label}<br/>Collar elevation: {elev_label}",
             "style": TOOLTIP_STYLE},
    description=f"{REGION}: wells below terrain",
)
legend = colorbar_html(f"{REGION}: wells below ground",
                       f"Vertical exaggeration x{VE}; lines drop from collar elevation by well depth",
                       "shallow", "deep", PLASMA,
                       "Line colour = source type (blue springs, purple boreholes, gold/red dug wells)<br/>"
                       "Basemap tiles &copy; Esri — sources include Esri, HERE, Garmin, FAO, NOAA, USGS, "
                       "OpenStreetMap contributors, and the GIS User Community")
print(f"{REGION}: {len(sub):,} records, {len(wells):,} wells drawn below ground")
show_and_export(deck_terrain, legend, PROJECT_DIR / "gw_terrain_wells_entire_area.html")

## 4. Monitoring-gap surface

Interpolating well depth would imply a water table, so this surface does something different: for every hexagon in Kenya, **height = distance to the nearest inventory record**. Tall, bright areas are where the inventory says nothing. It is derived entirely from the point locations, so it can't be misread as a groundwater quantity.

Set `GAP_USE_DEPTH_ONLY = True` to measure distance to the nearest record *with a measured depth* instead.

In [ ]:
GAP_USE_DEPTH_ONLY = False

def to_xyz(lat, lon, R=6371.0):
    lat, lon = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    return np.c_[R * np.cos(lat) * np.cos(lon), R * np.cos(lat) * np.sin(lon), R * np.sin(lat)]

src_pts = df[df["has_depth"]] if GAP_USE_DEPTH_ONLY else df
tree = cKDTree(to_xyz(src_pts["latitude"], src_pts["longitude"]))

try:
    nation = counties.geometry.union_all()
except AttributeError:               # older GeoPandas
    nation = counties.geometry.unary_union
cells = list(h3.geo_to_cells(nation.__geo_interface__ if hasattr(nation, "__geo_interface__") else nation, H3_RES))

centres = np.array([h3.cell_to_latlng(c) for c in cells])        # (lat, lon)
dist_km, _ = tree.query(to_xyz(centres[:, 0], centres[:, 1]))     # chord ~ great-circle at these scales

gap = pd.DataFrame({"hex": cells, "dist_km": dist_km})
g_hi = gap["dist_km"].quantile(0.97)
t_g = (gap["dist_km"] / max(g_hi, 1e-9)).clip(0, 1)
gap["h_gap"] = 2000 + (H_MAX - 2000) * t_g
gap["c_gap"] = [ramp(t, MAGMA) + [225] for t in t_g]
gap["dist_label"] = gap["dist_km"].map(lambda v: f"{v:,.1f} km")

deck_gap = pdk.Deck(
    layers=[pdk.Layer("H3HexagonLayer", gap, get_hexagon="hex", get_fill_color="c_gap",
                      get_elevation="h_gap", elevation_scale=1, extruded=True, coverage=0.95,
                      pickable=True, auto_highlight=True)],
    map_style=DARK_STYLE,
    tooltip={"html": "Nearest {} record: <b>{{dist_label}}</b>".format("depth" if GAP_USE_DEPTH_ONLY else "inventory"),
             "style": TOOLTIP_STYLE},
    initial_view_state=pdk.ViewState(latitude=MAP_CENTER["lat"], longitude=MAP_CENTER["lon"],
                                     zoom=5.3, pitch=55, bearing=-14),
    description="Kenya groundwater monitoring-gap surface",
)
legend = colorbar_html("Inventory gap",
                       "Distance from hex centre to nearest " + ("measured-depth well" if GAP_USE_DEPTH_ONLY else "inventory record"),
                       "0 km", f"{g_hi:,.0f}+ km", MAGMA,
                       "Not a groundwater quantity: it only shows where records are absent")
print(f"Median gap {gap['dist_km'].median():,.1f} km; 95th percentile {gap['dist_km'].quantile(.95):,.1f} km")
show_and_export(deck_gap, legend, PROJECT_DIR / "gw_monitoring_gap_3d.html")

## 5. GPU time filter with lonboard

Unlike the original month widget (which rebuilds the whole map on every change), lonboard uploads the data once and the slider only changes a GPU filter range, so scrubbing is instant. Choose **cumulative** (everything updated up to that month) or **only this month**.

`lonboard` and `pyarrow` are checked and installed by the setup cell. If this section was run before setup, restart the kernel and run the notebook from the top.

In [ ]:
from lonboard import Map, ScatterplotLayer, PolygonLayer
from lonboard.layer_extension import DataFilterExtension
import ipywidgets as widgets

months = sorted(df["month"].dropna().unique())
month_idx = df["month"].map({m: i for i, m in enumerate(months)}).to_numpy(dtype="float32")

pts = gpd.GeoDataFrame(
    df[["original_id", "name", "county", "Type"]].assign(month=df["month"], well_depth=df["height_value"]),
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326",
)
colors = np.array([TYPE_RGB.get(t, (145, 150, 160)) for t in df["Type"]], dtype=np.uint8)

hi_depth = df["height_value"].quantile(0.98)
dscale = (df["height_value"].clip(upper=hi_depth) / hi_depth).fillna(0).to_numpy()
radius = (900 + 2600 * np.sqrt(dscale)).astype("float32")     # metres; missing depth -> small base dot

point_layer = ScatterplotLayer.from_geopandas(
    pts, get_fill_color=colors, get_radius=radius,
    radius_min_pixels=2, radius_max_pixels=14, opacity=0.8,
    pickable=True, auto_highlight=True,
    extensions=[DataFilterExtension(filter_size=1)],
    get_filter_value=month_idx,
    filter_range=[0, len(months) - 1],
)
boundary_layer = PolygonLayer.from_geopandas(
    counties[["NAME_1", "geometry"]], filled=False, stroked=True,
    get_line_color=[120, 130, 140, 160], line_width_min_pixels=1,
)
time_map = Map(layers=[boundary_layer, point_layer],
               view_state={"longitude": MAP_CENTER["lon"], "latitude": MAP_CENTER["lat"], "zoom": 5.3})

slider = widgets.IntSlider(min=0, max=len(months) - 1, value=len(months) - 1, description="Month",
                           continuous_update=True, layout=widgets.Layout(width="92%"))
mode = widgets.ToggleButtons(options=["Cumulative", "Only this month"], value="Cumulative")
caption = widgets.HTML()

def update(_=None):
    i = slider.value
    cumulative = mode.value == "Cumulative"
    point_layer.filter_range = [0, i] if cumulative else [i, i]
    n = int((month_idx <= i).sum() if cumulative else (month_idx == i).sum())
    caption.value = f"<b>{months[i]}</b> — {n:,} records ({'cumulative' if cumulative else 'updated this month'})"

slider.observe(update, names="value")
mode.observe(update, names="value")
update()
display(widgets.VBox([mode, slider, caption, time_map]))

try:
    time_map.to_html(str(PROJECT_DIR / "gw_time_filter_lonboard.html"))
    print("Saved gw_time_filter_lonboard.html (export captures the initial filter state only)")
except Exception as exc:
    print("Standalone export skipped:", exc)

## 6. Cross-section along a transect

A fence-diagram style profile: wells within `BUFFER_KM` of a line, plotted by distance along the line. Floating bars run from collar elevation down to the well bottom; wells with no recorded depth show as collar markers only. Edit `A` and `B` (lon, lat) to cut any transect; by default it runs west–east through the middle of `REGION`.

In [ ]:
BUFFER_KM = 5.0
clat = float(sub["latitude"].mean())
A = (float(sub["longitude"].min()), clat)
B = (float(sub["longitude"].max()), clat)

lat0 = (A[1] + B[1]) / 2
kx, ky = 111.32 * np.cos(np.radians(lat0)), 110.57          # km per degree
def to_km(lon, lat):
    return np.c_[(np.asarray(lon) - A[0]) * kx, (np.asarray(lat) - A[1]) * ky]

p = to_km(sub["longitude"], sub["latitude"])
ab = to_km([B[0]], [B[1]])[0]
L = float(np.hypot(*ab))
u = ab / L
sub["along_km"] = p @ u
sub["off_km"] = np.abs(p[:, 0] * u[1] - p[:, 1] * u[0])

sect = sub[(sub["off_km"] <= BUFFER_KM) & sub["along_km"].between(0, L)].sort_values("along_km")
print(f"{REGION}: {len(sect):,} records within {BUFFER_KM:g} km of a {L:,.0f} km transect")

fig = go.Figure()
fig.add_trace(go.Scatter(x=sect["along_km"], y=sect["elev_m"], mode="lines",
                         line=dict(color="#8a97a6", width=1.5, dash="dot"), name="Collar elevation profile"))
d = sect[sect["has_depth"] & ~sect["depth_review_flag"]]
for t, g in d.groupby("Type"):
    r, gg, b = TYPE_RGB.get(t, (145, 150, 160))
    fig.add_trace(go.Bar(x=g["along_km"], y=g["well_depth"], base=g["elev_m"] - g["well_depth"],
                         width=max(L / 220, 0.15), name=t, marker_color=f"rgb({r},{gg},{b})",
                         customdata=np.c_[g["name"], g["well_depth"], g["off_km"]],
                         hovertemplate="%{customdata[0]}<br>depth %{customdata[1]:.1f} m<br>%{customdata[2]:.1f} km off line<extra></extra>"))
nod = sect[~sect["has_depth"]]
fig.add_trace(go.Scatter(x=nod["along_km"], y=nod["elev_m"], mode="markers", name="No depth recorded",
                         marker=dict(symbol="circle-open", size=7, color="#aab4c0"),
                         hovertemplate="%{text}<extra></extra>", text=nod["name"]))
fig.update_layout(template="plotly_white", barmode="overlay", height=520,
                  title=f"{REGION}: cross-section, west to east (true scale axes)",
                  xaxis_title="Distance along transect (km)", yaxis_title="Elevation (m a.s.l.)",
                  legend=dict(orientation="h", y=1.1))
fig.show()
fig.write_html(PROJECT_DIR / "gw_cross_section.html", include_plotlyjs=True)

## 7. Depth distributions (not just medians)

In [ ]:
ok = df[df["has_depth"] & ~df["depth_review_flag"]]

fig_type = px.violin(ok, x="Type", y="well_depth", color="Type", box=True, points=False, log_y=True,
                     color_discrete_map={k: f"rgb{v}" for k, v in TYPE_RGB.items()},
                     title="Well depth by source type (log scale; records >1,000 m excluded)")
fig_type.update_layout(template="plotly_white", showlegend=False, yaxis_title="Well depth (m)", height=480)
fig_type.show()

top = ok["county"].value_counts().head(20)
tc = ok[ok["county"].isin(top.index)].copy()
tc["label"] = tc["county"].map(lambda c: f"{c} (n={top[c]})")
order = tc.groupby("label")["well_depth"].median().sort_values().index
fig_cty = px.box(tc, x="label", y="well_depth", log_y=True, category_orders={"label": list(order)},
                 title="Well depth by county — 20 best-sampled counties, with sample sizes")
fig_cty.update_layout(template="plotly_white", xaxis_title="", yaxis_title="Well depth (m)",
                      xaxis_tickangle=-40, height=540)
fig_cty.show()
fig_type.write_html(PROJECT_DIR / "gw_depth_by_type.html", include_plotlyjs="cdn")
fig_cty.write_html(PROJECT_DIR / "gw_depth_by_county.html", include_plotlyjs="cdn")